# CASML B0 — prepare

Chỉ chạy bước prepare. Chỉnh đường dẫn artifact đã có trong cell cấu hình.

In [ ]:
from pathlib import Path
import shutil, subprocess, sys

# Đặt đường dẫn thư mục có pyproject.toml nếu có nhiều bản project.
ROOT_OVERRIDE = None

if ROOT_OVERRIDE:
    source = Path(ROOT_OVERRIDE).resolve()
else:
    local = [Path.cwd(), *Path.cwd().parents]
    candidates = [p for p in local if (p / "casml_b0/cli.py").is_file()]
    if not candidates and Path("/kaggle/input").exists():
        candidates = [p.parent for p in Path("/kaggle/input").rglob("pyproject.toml")
                      if (p.parent / "casml_b0/cli.py").is_file()]
    candidates = list(dict.fromkeys(p.resolve() for p in candidates))
    if len(candidates) != 1:
        raise RuntimeError(f"Tìm thấy {len(candidates)} project; hãy đặt ROOT_OVERRIDE đúng thư mục đã giải nén.")
    source = candidates[0]
if not (source / "casml_b0/cli.py").is_file():
    raise FileNotFoundError("ROOT_OVERRIDE chưa trỏ tới thư mục project")
if Path("/kaggle/working").exists():
    ROOT = Path("/kaggle/working/casml_b0_starter")
    if source != ROOT and not ROOT.exists():
        shutil.copytree(source, ROOT, ignore=shutil.ignore_patterns("__pycache__", "artifacts", "runs", "outputs", ".venv"))
    # Giữ mã người dùng đã sửa ở Working khi chạy lại cell.
    WORK = Path("/kaggle/working/casml_b0_work")
else:
    ROOT, WORK = source, source
WORK.mkdir(parents=True, exist_ok=True)
print("Project:", ROOT)
print("Artifacts:", WORK)

def stage(*args):
    subprocess.run([sys.executable, "-m", "casml_b0", *map(str, args)], cwd=ROOT, check=True)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements-prepare.txt")], check=True)


In [ ]:
# SỬA đường dẫn cho đúng dữ liệu của bạn. Không dùng PDF overview làm sách.
DATA = Path("/kaggle/input/casml-generative-ai-hackathon/Dataset_RAG (1)") if Path("/kaggle/input").exists() else ROOT / "data"
PDF = DATA / "book.pdf"
QUERIES = DATA / "queries.json"
SAMPLE = None  # Đặt Path đến sample_submission.csv chính thức nếu có.
PAGE_MAP = None  # Đặt Path đến page_map_override.csv sau khi kiểm tra số trang/mục.
CORPUS = WORK / "artifacts/corpus_v1"
INDEX = WORK / "artifacts/index_v1"
RETRIEVAL = WORK / "artifacts/retrieval_v1"
RUN = WORK / "runs/b0_g01"
OUTPUT = WORK / "outputs/b0_g01"
GEN_CONFIG = ROOT / "configs/generate.yaml"
# Khi đổi prompt/model/config, dùng RUN và OUTPUT mới.


In [ ]:
args = ["prepare", "--pdf", PDF, "--config", ROOT / "configs/prepare.yaml", "--out", CORPUS]
if PAGE_MAP is not None:
    args += ["--page-map-override", PAGE_MAP]
stage(*args)
print((CORPUS / "report.json").read_text())
